# Module 09: Multimodal Models (Mini-CLIP)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/09-multimodal/notebook.ipynb)

**GPU recommended:** Yes (CLIP training benefits from GPU, ~5 min on CPU).

## 1. Overview

CLIP (Contrastive Language-Image Pretraining) learns a shared embedding space for images and text
by training dual encoders with a contrastive objective. In this notebook we build a minimal version
from scratch:

- **Image encoder**: small CNN operating on MNIST 28x28 grayscale images.
- **Text encoder**: small transformer encoder operating on tokenized digit names.
- **Contrastive loss**: InfoNCE, which pushes matching (image, text) pairs together and
  non-matching pairs apart in the shared embedding space.

We train on MNIST digits paired with their text labels ("zero" through "nine"), then
demonstrate zero-shot classification, cross-modal retrieval, and embedding-space visualization.

## 2. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import math

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

plt.style.use("seaborn-v0_8-whitegrid")

## 3. Vocabulary and Tokenization

We define a tiny vocabulary for digit names and build a simple tokenizer that converts
each character to an integer index. A `[PAD]` token is included for batching.

In [ ]:
DIGIT_NAMES = ["zero", "one", "two", "three", "four",
               "five", "six", "seven", "eight", "nine"]

# Build character-level vocabulary
all_chars = sorted(set("".join(DIGIT_NAMES)))
char_to_idx = {"[PAD]": 0}
for i, ch in enumerate(all_chars, start=1):
    char_to_idx[ch] = i

VOCAB_SIZE = len(char_to_idx)
MAX_TEXT_LEN = max(len(name) for name in DIGIT_NAMES)

print(f"Vocabulary ({VOCAB_SIZE} tokens): {char_to_idx}")
print(f"Max text length: {MAX_TEXT_LEN}")


def tokenize(text, max_len=MAX_TEXT_LEN):
    """Convert a string to a padded tensor of character indices."""
    ids = [char_to_idx[ch] for ch in text]
    ids += [0] * (max_len - len(ids))  # pad
    return torch.tensor(ids, dtype=torch.long)

## 4. Toy Image-Text Dataset

We wrap MNIST so that each sample returns `(image, tokenized_text_label, class_index)`.
The text label is the English word for the digit (e.g. 3 -> "three").

In [ ]:
class MNISTTextDataset(Dataset):
    """MNIST images paired with tokenized digit-name text labels."""

    def __init__(self, train=True):
        self.mnist = datasets.MNIST(
            root="./data", train=train, download=True,
            transform=transforms.ToTensor(),
        )

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        image, label = self.mnist[idx]
        text_tokens = tokenize(DIGIT_NAMES[label])
        return image, text_tokens, label


train_dataset = MNISTTextDataset(train=True)
test_dataset = MNISTTextDataset(train=False)

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples:     {len(test_dataset):,}")

### 4.1 Inspect a Batch

In [ ]:
images, texts, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(DIGIT_NAMES[labels[i].item()])
    ax.axis("off")
fig.suptitle("Sample Image-Text Pairs", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Model Architecture

### 5.1 Image Encoder (CNN)

A small convolutional network that maps a 1x28x28 image to an embedding vector of
dimension `embed_dim`.

In [ ]:
class ImageEncoder(nn.Module):
    """Small CNN encoder for 28x28 grayscale images."""

    def __init__(self, embed_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # -> 32 x 14 x 14
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # -> 64 x 7 x 7
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),  # -> 128 x 1 x 1
        )
        self.projection = nn.Linear(128, embed_dim)

    def forward(self, x):
        x = self.conv(x).squeeze(-1).squeeze(-1)  # (B, 128)
        x = self.projection(x)                     # (B, embed_dim)
        return x

### 5.2 Text Encoder (Transformer)

A small transformer encoder that processes character-level token sequences and outputs
a fixed-size embedding (via mean pooling over non-padded positions).

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""

    def __init__(self, d_model, max_len=64):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
class TextEncoder(nn.Module):
    """Small transformer encoder for character-level text."""

    def __init__(self, vocab_size, embed_dim=128, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128,
            dropout=0.1, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.projection = nn.Linear(d_model, embed_dim)

    def forward(self, tokens):
        # tokens: (B, seq_len) with 0 = padding
        pad_mask = tokens == 0  # True where padded
        x = self.embedding(tokens)
        x = self.pos_enc(x)
        x = self.transformer(x, src_key_padding_mask=pad_mask)

        # Mean pool over non-padded positions
        mask = (~pad_mask).unsqueeze(-1).float()  # (B, seq_len, 1)
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        x = self.projection(x)  # (B, embed_dim)
        return x

### 5.3 Mini-CLIP Model

Combines both encoders and a learnable temperature parameter for the contrastive loss.

In [ ]:
class MiniCLIP(nn.Module):
    """Contrastive Language-Image Pretraining (mini version)."""

    def __init__(self, vocab_size, embed_dim=128):
        super().__init__()
        self.image_encoder = ImageEncoder(embed_dim=embed_dim)
        self.text_encoder = TextEncoder(vocab_size=vocab_size, embed_dim=embed_dim)
        # Learnable temperature (log-parameterized for stability)
        self.log_temperature = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def encode_image(self, images):
        return F.normalize(self.image_encoder(images), dim=-1)

    def encode_text(self, tokens):
        return F.normalize(self.text_encoder(tokens), dim=-1)

    def forward(self, images, tokens):
        image_emb = self.encode_image(images)
        text_emb = self.encode_text(tokens)
        return image_emb, text_emb, self.log_temperature.exp()


model = MiniCLIP(vocab_size=VOCAB_SIZE, embed_dim=128).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

## 6. Contrastive Loss (InfoNCE)

InfoNCE treats each (image, text) pair in a batch as the positive pair and all other
cross-modal combinations as negatives. The loss is symmetric: we compute a cross-entropy
loss from images-to-texts and from texts-to-images, then average.

In [ ]:
def info_nce_loss(image_emb, text_emb, temperature):
    """
    Symmetric InfoNCE (CLIP) loss.

    Args:
        image_emb: (B, D) L2-normalized image embeddings.
        text_emb:  (B, D) L2-normalized text embeddings.
        temperature: scalar temperature.

    Returns:
        Scalar loss (average of image->text and text->image CE).
    """
    # Cosine similarity matrix scaled by temperature
    logits = (image_emb @ text_emb.T) * temperature  # (B, B)
    labels = torch.arange(logits.size(0), device=logits.device)

    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2


# Quick sanity check with random embeddings
_img = F.normalize(torch.randn(4, 128), dim=-1)
_txt = F.normalize(torch.randn(4, 128), dim=-1)
print(f"Sanity check loss (random): {info_nce_loss(_img, _txt, temperature=14.3).item():.4f}")
print(f"Expected ~ -ln(1/B) = {math.log(4):.4f}")

## 7. Training Loop

In [ ]:
EPOCHS = 10
LR = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
train_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    num_batches = 0

    for images, tokens, _ in train_loader:
        images = images.to(device)
        tokens = tokens.to(device)

        image_emb, text_emb, temperature = model(images, tokens)
        loss = info_nce_loss(image_emb, text_emb, temperature)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    scheduler.step()
    avg_loss = epoch_loss / num_batches
    train_losses.append(avg_loss)
    temp_val = model.log_temperature.exp().item()
    print(f"Epoch {epoch:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Temp: {temp_val:.2f} | LR: {scheduler.get_last_lr()[0]:.2e}")

### 7.1 Training Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, EPOCHS + 1), train_losses, marker="o", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("InfoNCE Loss")
ax.set_title("Training Loss")
ax.set_xticks(range(1, EPOCHS + 1))
plt.tight_layout()
plt.show()

## 8. Embedding Extraction

We extract normalized embeddings for the test set and for all 10 digit-name texts.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader, max_samples=2000):
    """Extract image and text embeddings from a DataLoader."""
    model.eval()
    all_img_emb, all_txt_emb, all_labels = [], [], []
    count = 0

    for images, tokens, labels in loader:
        images = images.to(device)
        tokens = tokens.to(device)

        img_emb = model.encode_image(images)
        txt_emb = model.encode_text(tokens)

        all_img_emb.append(img_emb.cpu())
        all_txt_emb.append(txt_emb.cpu())
        all_labels.append(labels)

        count += images.size(0)
        if count >= max_samples:
            break

    return (
        torch.cat(all_img_emb)[:max_samples],
        torch.cat(all_txt_emb)[:max_samples],
        torch.cat(all_labels)[:max_samples],
    )


img_emb, txt_emb, labels = extract_embeddings(model, test_loader, max_samples=2000)
print(f"Extracted {img_emb.shape[0]} image and text embeddings of dim {img_emb.shape[1]}")

In [ ]:
# Compute text embeddings for all 10 digit names (used for zero-shot classification)
all_text_tokens = torch.stack([tokenize(name) for name in DIGIT_NAMES]).to(device)

with torch.no_grad():
    model.eval()
    class_text_emb = model.encode_text(all_text_tokens).cpu()

print(f"Class text embeddings shape: {class_text_emb.shape}")

## 9. Visualize the Shared Embedding Space (t-SNE)

We project both image and text embeddings into 2D with t-SNE. Matching image-text pairs
should cluster together if the model has learned a good shared representation.

In [ ]:
# Use a subset for t-SNE (faster and cleaner plot)
N_VIS = 500
vis_img = img_emb[:N_VIS]
vis_txt = class_text_emb  # 10 class text embeddings
vis_labels = labels[:N_VIS]

# Combine image embeddings with the 10 class text embeddings
combined = torch.cat([vis_img, vis_txt], dim=0).numpy()
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
coords = tsne.fit_transform(combined)

img_coords = coords[:N_VIS]
txt_coords = coords[N_VIS:]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.tab10

# Plot image embeddings as small dots
for digit in range(10):
    mask = vis_labels.numpy() == digit
    ax.scatter(
        img_coords[mask, 0], img_coords[mask, 1],
        c=[cmap(digit)], s=10, alpha=0.5, label=f"{digit} (image)",
    )

# Plot text embeddings as large stars
for digit in range(10):
    ax.scatter(
        txt_coords[digit, 0], txt_coords[digit, 1],
        c=[cmap(digit)], s=300, marker="*", edgecolors="black", linewidths=0.8,
    )
    ax.annotate(
        DIGIT_NAMES[digit], (txt_coords[digit, 0], txt_coords[digit, 1]),
        fontsize=9, fontweight="bold", ha="center", va="bottom",
        xytext=(0, 8), textcoords="offset points",
    )

ax.set_title("t-SNE of Shared Embedding Space (dots=images, stars=text)", fontsize=13)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, markerscale=2)
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
plt.tight_layout()
plt.show()

## 10. Zero-Shot Classification

Given an unseen test image, we compute its cosine similarity to each of the 10 text
embeddings and pick the most similar one. No classifier head is needed -- the shared
embedding space enables this directly.

In [ ]:
def zero_shot_classify(image_embeddings, class_text_embeddings):
    """Classify images by cosine similarity to class text embeddings."""
    # Both inputs are already L2-normalized
    similarities = image_embeddings @ class_text_embeddings.T  # (N, 10)
    predictions = similarities.argmax(dim=-1)
    return predictions, similarities

In [ ]:
predictions, similarities = zero_shot_classify(img_emb, class_text_emb)
accuracy = (predictions == labels).float().mean().item()
print(f"Zero-shot classification accuracy on {img_emb.shape[0]} test images: {accuracy:.1%}")

In [ ]:
# Per-class accuracy breakdown
print("\nPer-class accuracy:")
print("-" * 30)
for digit in range(10):
    mask = labels == digit
    if mask.sum() > 0:
        class_acc = (predictions[mask] == digit).float().mean().item()
        print(f"  {DIGIT_NAMES[digit]:>5s} ({digit}): {class_acc:.1%}  (n={mask.sum().item()})")

### 10.1 Example Predictions

In [ ]:
# Show some example predictions
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
test_images = []
for i in range(10):
    img, _, _ = test_dataset[i]
    test_images.append(img)

for i, ax in enumerate(axes.flat):
    ax.imshow(test_images[i].squeeze(), cmap="gray")
    true_label = labels[i].item()
    pred_label = predictions[i].item()
    color = "green" if true_label == pred_label else "red"
    ax.set_title(f"True: {DIGIT_NAMES[true_label]}\nPred: {DIGIT_NAMES[pred_label]}",
                 fontsize=9, color=color)
    ax.axis("off")

fig.suptitle("Zero-Shot Classification Examples (green=correct, red=wrong)", fontsize=13)
plt.tight_layout()
plt.show()

## 11. Cross-Modal Retrieval

### 11.1 Text-to-Image Retrieval

Given a text query (e.g. "seven"), find the test images whose embeddings are closest.

In [ ]:
def text_to_image_retrieval(query_text, image_embeddings, top_k=5):
    """Retrieve top-k images closest to a text query."""
    tokens = tokenize(query_text).unsqueeze(0).to(device)
    with torch.no_grad():
        model.eval()
        query_emb = model.encode_text(tokens).cpu()
    sims = (query_emb @ image_embeddings.T).squeeze(0)
    top_indices = sims.argsort(descending=True)[:top_k]
    return top_indices, sims[top_indices]

In [ ]:
query_words = ["three", "seven", "zero"]
TOP_K = 6

fig, axes = plt.subplots(len(query_words), TOP_K, figsize=(12, 2.2 * len(query_words)))

for row, query in enumerate(query_words):
    indices, scores = text_to_image_retrieval(query, img_emb, top_k=TOP_K)
    for col in range(TOP_K):
        idx = indices[col].item()
        img, _, lbl = test_dataset[idx]
        ax = axes[row, col]
        ax.imshow(img.squeeze(), cmap="gray")
        ax.set_title(f"sim={scores[col]:.3f}", fontsize=8)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(f'Query: "{query}"', fontsize=10, rotation=0, labelpad=70, va="center")

fig.suptitle("Text-to-Image Retrieval (top-6 results)", fontsize=13)
plt.tight_layout()
plt.show()

### 11.2 Image-to-Text Retrieval

Given a query image, rank all 10 digit-name text embeddings by similarity.

In [ ]:
def image_to_text_retrieval(image_embedding, class_text_embeddings):
    """Rank all text labels by cosine similarity to an image embedding."""
    sims = (image_embedding @ class_text_embeddings.T).squeeze(0)
    ranking = sims.argsort(descending=True)
    return ranking, sims

In [ ]:
# Pick a few test images and show their text ranking
query_indices = [0, 15, 42, 99, 200]

fig, axes = plt.subplots(1, len(query_indices), figsize=(14, 3))

for col, qi in enumerate(query_indices):
    ax = axes[col]
    img_tensor, _, true_label = test_dataset[qi]
    ax.imshow(img_tensor.squeeze(), cmap="gray")
    ax.axis("off")

    ranking, sims = image_to_text_retrieval(img_emb[qi:qi+1], class_text_emb)
    top3 = [(DIGIT_NAMES[r.item()], sims[r].item()) for r in ranking[:3]]
    title_lines = [f"True: {DIGIT_NAMES[true_label]}"]
    for rank, (name, score) in enumerate(top3, 1):
        title_lines.append(f"#{rank}: {name} ({score:.3f})")
    ax.set_title("\n".join(title_lines), fontsize=8)

fig.suptitle("Image-to-Text Retrieval (top-3 matches)", fontsize=13)
plt.tight_layout()
plt.show()

## 12. NxN Similarity Matrix Visualization

We take a batch of images and texts and visualize the full similarity matrix. In a
well-trained model, the diagonal (matching pairs) should have the highest values.

In [ ]:
# Select one image per digit class for a clean 10x10 matrix
selected_indices = []
for digit in range(10):
    for i in range(len(labels)):
        if labels[i].item() == digit:
            selected_indices.append(i)
            break

sel_img_emb = img_emb[selected_indices]   # (10, 128)
sel_labels = labels[selected_indices]       # (10,)

# Similarity: each selected image vs. each class text
sim_matrix = (sel_img_emb @ class_text_emb.T).numpy()  # (10, 10)

print(f"Similarity matrix shape: {sim_matrix.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_matrix, cmap="viridis", aspect="auto", vmin=-1, vmax=1)

# Annotate each cell with its value
for i in range(10):
    for j in range(10):
        color = "white" if sim_matrix[i, j] < 0.5 else "black"
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center",
                fontsize=8, color=color)

image_labels = [f"img {sel_labels[i].item()}" for i in range(10)]
text_labels = [DIGIT_NAMES[i] for i in range(10)]

ax.set_xticks(range(10))
ax.set_xticklabels(text_labels, rotation=45, ha="right")
ax.set_yticks(range(10))
ax.set_yticklabels(image_labels)
ax.set_xlabel("Text Labels")
ax.set_ylabel("Image Samples")
ax.set_title("Image-Text Cosine Similarity Matrix (10x10)")
fig.colorbar(im, ax=ax, label="Cosine Similarity")
plt.tight_layout()
plt.show()

## 13. Retrieval Metrics

We compute Recall@1 and Recall@5 for both text-to-image and image-to-text retrieval
on a larger subset of the test set.

In [ ]:
def compute_retrieval_metrics(image_embeddings, text_embeddings, labels, k_values=(1, 5)):
    """
    Compute Recall@k for image-to-text and text-to-image retrieval.
    A retrieval is correct if the retrieved item has the same class label.
    """
    sim_matrix = image_embeddings @ text_embeddings.T  # (N, N)
    n = sim_matrix.size(0)

    results = {}
    for direction, sims in [("i2t", sim_matrix), ("t2i", sim_matrix.T)]:
        for k in k_values:
            top_k_indices = sims.topk(k, dim=-1).indices  # (N, k)
            correct = 0
            for i in range(n):
                retrieved_labels = labels[top_k_indices[i]]
                if labels[i] in retrieved_labels:
                    correct += 1
            results[f"{direction}_R@{k}"] = correct / n

    return results


metrics = compute_retrieval_metrics(img_emb, txt_emb, labels)
print("Cross-modal retrieval metrics:")
print("-" * 35)
for name, value in metrics.items():
    direction = "Image->Text" if name.startswith("i2t") else "Text->Image"
    k_part = name.split("_")[-1]
    print(f"  {direction} {k_part}: {value:.1%}")

## 14. Embedding Space Analysis

We examine how well the model aligns modalities by computing the average cosine
similarity between matching and non-matching pairs.

In [ ]:
# Compute average similarity for matching vs. non-matching pairs
sim_all = (img_emb @ txt_emb.T).numpy()
n = sim_all.shape[0]

matching_sims = []
non_matching_sims = []

for i in range(n):
    for j in range(n):
        if labels[i] == labels[j]:
            matching_sims.append(sim_all[i, j])
        else:
            non_matching_sims.append(sim_all[i, j])

# Only sample if there are too many pairs
if len(non_matching_sims) > 100000:
    non_matching_sims = list(np.random.choice(non_matching_sims, 100000, replace=False))

print(f"Avg cosine sim (matching pairs):     {np.mean(matching_sims):.4f}")
print(f"Avg cosine sim (non-matching pairs):  {np.mean(non_matching_sims):.4f}")
print(f"Gap:                                  {np.mean(matching_sims) - np.mean(non_matching_sims):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(matching_sims, bins=50, alpha=0.7, label="Matching pairs", density=True)
ax.hist(non_matching_sims, bins=50, alpha=0.7, label="Non-matching pairs", density=True)
ax.set_xlabel("Cosine Similarity")
ax.set_ylabel("Density")
ax.set_title("Distribution of Cross-Modal Cosine Similarities")
ax.legend()
plt.tight_layout()
plt.show()

## 15. PCA Visualization (Alternative to t-SNE)

PCA gives a deterministic, global view of the embedding structure.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
combined_pca = torch.cat([vis_img, vis_txt], dim=0).numpy()
coords_pca = pca.fit_transform(combined_pca)

img_pca = coords_pca[:N_VIS]
txt_pca = coords_pca[N_VIS:]

explained = pca.explained_variance_ratio_
print(f"PCA explained variance: PC1={explained[0]:.1%}, PC2={explained[1]:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.tab10

for digit in range(10):
    mask = vis_labels.numpy() == digit
    ax.scatter(
        img_pca[mask, 0], img_pca[mask, 1],
        c=[cmap(digit)], s=10, alpha=0.5, label=f"{digit} (image)",
    )

for digit in range(10):
    ax.scatter(
        txt_pca[digit, 0], txt_pca[digit, 1],
        c=[cmap(digit)], s=300, marker="*", edgecolors="black", linewidths=0.8,
    )
    ax.annotate(
        DIGIT_NAMES[digit], (txt_pca[digit, 0], txt_pca[digit, 1]),
        fontsize=9, fontweight="bold", ha="center", va="bottom",
        xytext=(0, 8), textcoords="offset points",
    )

ax.set_title("PCA of Shared Embedding Space (dots=images, stars=text)", fontsize=13)
ax.set_xlabel(f"PC1 ({explained[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({explained[1]:.1%} variance)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, markerscale=2)
plt.tight_layout()
plt.show()

## 16. Summary

In this notebook we built a **mini-CLIP** model from scratch with:

1. **Dual encoders** -- a small CNN for images and a small transformer for text -- projecting
   both modalities into a shared 128-dimensional embedding space.
2. **InfoNCE contrastive loss** that aligns matching image-text pairs while pushing
   non-matching pairs apart.
3. **Zero-shot classification** by comparing image embeddings to text embeddings of class
   names, without any task-specific classifier.
4. **Cross-modal retrieval** in both directions (text-to-image and image-to-text).
5. **Visualizations** (t-SNE, PCA, similarity matrix, similarity distribution) confirming
   that the model learns meaningful cross-modal representations.

Key takeaways:
- Contrastive learning can align representations across modalities with no explicit labels
  at test time.
- The learnable temperature parameter controls the sharpness of the similarity distribution.
- Even on a simple dataset like MNIST, the CLIP-style approach demonstrates strong
  zero-shot transfer capabilities.